In [ ]:
# MOSS-TTS against the same nine speakers, sentences, seed and reference clips.
# transformers is pinned because MOSS's remote code calls generation internals
# that 5.1 removed; upstream's card asks for 5.0.0 for the same reason.
!pip -q install "transformers>=5.0,<5.1" accelerate speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
import transformers

INPUT = Path("/kaggle/input")
print("mounted     :", sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else "nothing")

# The dataset mounts at /kaggle/input/datasets/<owner>/<slug>/, so the repo is
# found by a file that has to be inside it rather than by an assumed depth.
found = sorted(INPUT.rglob("repo/scripts/benchmark.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo under /kaggle/input - see the directories above")
SOURCE = found[0].parent.parent
print("source      :", SOURCE)

REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SOURCE, REPO)

version = tuple(int(part) for part in transformers.__version__.split(".")[:2])
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
print("gpus        :", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
# T4 is Turing and has no native bfloat16, so the engine falls back to float16.
# Printed rather than assumed, because it decides what precision MOSS runs at.
print("bf16 native :", torch.cuda.is_bf16_supported())

assert (5, 0) <= version < (5, 1), f"MOSS needs transformers 5.0.x, found {transformers.__version__}"
assert torch.cuda.device_count() >= 1, "no GPU - set the accelerator"


def run(*arguments):
    command = [sys.executable, "scripts/benchmark.py", *arguments]
    print(" ".join(command), flush=True)
    outcome = subprocess.run(command, cwd=REPO, text=True)
    print("exit code:", outcome.returncode, flush=True)
    return outcome.returncode


In [ ]:
# Prove the scale before measuring anything on it.
#
# Every number on record was produced with transformers 5.14. This kernel runs
# 5.0.x, so its embedder and recogniser are a different build. Re-score renders
# whose answer is already established and require that answer back. If the
# scale has moved, stop here - a MOSS number from a shifted scale would be
# worse than no MOSS number, because it would look comparable and not be.
import json

assert run("--label", "baseline-moss-env", "--score-only", "--replay-from", "baseline",
           "--device", "cuda") == 0

summary = json.loads((REPO / "eval/results/baseline-moss-env.json").read_text())["summary"]
print("control held-out:", summary["held_out_mean"], " CER:", summary["cer_mean"], flush=True)
assert abs(summary["held_out_mean"] - 0.5196) < 0.0005, (
    f"this environment scores the same audio at {summary['held_out_mean']:.4f}, "
    "not 0.5196 - its numbers are not on the established scale"
)
print("scale confirmed", flush=True)


In [ ]:
# Same flags as every other engine apart from the device, so the cells line up.
assert run("--label", "moss", "--engine", "moss", "--device", "cuda") == 0

# benchmark.py records failures in its results file, not in its exit code: an
# earlier run exited 0 with all 27 cells failed and the kernel said COMPLETE.
summary = json.loads((REPO / "eval/results/moss.json").read_text())["summary"]
print("moss summary:", summary, flush=True)
failures = summary.get("failures") or 0
assert failures == 0, f"MOSS failed {failures} cells - see the traceback above"
assert (summary.get("renders") or 0) > 0, "MOSS produced no renders"


In [ ]:
destination = Path("/kaggle/working/results")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)

renders = Path("/kaggle/working/moss_renders")
renders.mkdir(exist_ok=True)
for path in (REPO / "outputs/benchmark/moss").glob("*.wav"):
    shutil.copy2(path, renders / path.name)
print("renders saved:", len(list(renders.glob("*.wav"))))

for name in ("moss", "baseline-moss-env"):
    path = destination / f"{name}.json"
    if path.is_file():
        data = json.loads(path.read_text())["summary"]
        print(name, {k: data.get(k) for k in ("held_out_mean", "cer_mean", "renders", "failures")})
